# Feature Engineering


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *

# Data Loading 


In [2]:
FILENAME = "stock_data_with_features.parquet"

In [3]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

Data loaded for ['AAPL']
Data loaded for [['SPY', 'QQQ']]


In [4]:
FE = FeatureEngineer(market_data, benchmark_data, filename=FILENAME)
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600
...,...,...,...,...,...,...,...,...
2509,2026-11-09,AAPL,Technology,350.423737,352.062476,344.518220,344.738054,48753100
2510,2026-11-10,AAPL,Technology,348.245392,353.601307,346.196930,351.083232,29756100
2511,2026-11-11,AAPL,Technology,355.359985,355.639772,349.064750,349.284585,34453500
2512,2026-11-12,AAPL,Technology,354.880005,359.369995,354.410004,357.809998,31875000


Benchmark Features

In [5]:
list(benchmark_data["Ticker"].unique())

['QQQ', 'SPY']

# Returns for the benchmark

In [6]:
FE.add_benchmark_returns()

FE.data


Rows: 2514
Duplicate ticker/date pairs: 0


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237
...,...,...,...,...,...,...,...,...,...,...
2509,2026-11-09,AAPL,Technology,350.423737,352.062476,344.518220,344.738054,48753100,0.035437,0.023664
2510,2026-11-10,AAPL,Technology,348.245392,353.601307,346.196930,351.083232,29756100,-0.006756,-0.003680
2511,2026-11-11,AAPL,Technology,355.359985,355.639772,349.064750,349.284585,34453500,0.023149,0.013188
2512,2026-11-12,AAPL,Technology,354.880005,359.369995,354.410004,357.809998,31875000,-0.003201,-0.003540


In [7]:
print("Rows:", len(FE.data))

print(
    "Duplicate ticker/date pairs:",
    FE.data.duplicated(
        ["Ticker", "Date"]
    ).sum()
)

print(
    FE.data.groupby("Ticker")["Date"]
    .nunique()
    .describe()
)

Rows: 2514
Duplicate ticker/date pairs: 0
count       1.0
mean     2514.0
std         NaN
min      2514.0
25%      2514.0
50%      2514.0
75%      2514.0
max      2514.0
Name: Date, dtype: float64


# Returns

In [8]:
FE.add_daily_returns() #Daily Price Movement 
FE.add_return_feature() #Very short term momentum - default is 5 days. 
FE.add_return_feature(10)
FE.add_return_feature(20)
FE.add_return_feature(60)

FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,Daily_Return,Return_5D,Return_10D,Return_20D,Return_60D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803,0.013244,NaN,NaN,NaN,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878,0.026888,NaN,NaN,NaN,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141,-0.000364,NaN,NaN,NaN,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237,0.001000,NaN,NaN,NaN,NaN


# Volatility Features

In [9]:

FE.add_volatility_feature() #recent variability - default is 20 days
FE.add_volatility_feature(5) 
FE.add_volatility_feature(10) #recent variability - default is 20 days
FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,Daily_Return,Return_5D,Return_10D,Return_20D,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803,0.013244,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878,0.026888,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141,-0.000364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237,0.001000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Trend Features

In [10]:
FE.add_SMA_feature() #recent price trend - default is 20 
FE.add_SMA_feature(50) #longer term price trend 
FE.add_price_relative_to_sma() #position relative to trend - default is 20
FE.add_price_relative_to_sma(50) #position relative to longer trend 

FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_10D,Return_20D,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Volume Features

In [11]:
FE.add_average_volume_feature() #Trading activity - default is 20

FE.add_relative_volume()

FE.data.head()

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509,2026-11-09,AAPL,Technology,350.423737,352.062476,344.518220,344.738054,48753100,0.035437,0.023664,...,0.146308,0.016953,0.030577,0.023231,343.609422,336.378015,1.019832,1.041756,49839655.0,0.978199
2510,2026-11-10,AAPL,Technology,348.245392,353.601307,346.196930,351.083232,29756100,-0.006756,-0.003680,...,0.140450,0.017042,0.028251,0.023223,343.971147,337.010734,1.012426,1.033336,49378270.0,0.602615
2511,2026-11-11,AAPL,Technology,355.359985,355.639772,349.064750,349.284585,34453500,0.023149,0.013188,...,0.147083,0.017565,0.017625,0.023699,344.604166,337.620271,1.031212,1.052543,48597890.0,0.708951
2512,2026-11-12,AAPL,Technology,354.880005,359.369995,354.410004,357.809998,31875000,-0.003201,-0.003540,...,0.120947,0.017548,0.014763,0.023403,345.261150,338.223606,1.027860,1.049247,48203275.0,0.661262


# Target Variable - 5 Day return

In [13]:
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Return_60D,Volatility_20D,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D
0,2016-11-14,AAPL,Technology,24.280031,24.762370,23.905645,24.739402,204702000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-15,AAPL,Technology,24.601593,24.732514,24.383393,24.477563,129058000,0.013871,0.007803,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-11-16,AAPL,Technology,25.263075,25.318201,24.484442,24.507410,235362000,0.005937,-0.001878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-11-17,AAPL,Technology,25.253891,25.345765,24.996645,25.221735,110528000,0.007271,0.005141,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-11-18,AAPL,Technology,25.279156,25.389405,25.187283,25.201064,113715600,-0.003737,-0.002237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509,2026-11-09,AAPL,Technology,350.423737,352.062476,344.518220,344.738054,48753100,0.035437,0.023664,...,0.146308,0.016953,0.030577,0.023231,343.609422,336.378015,1.019832,1.041756,49839655.0,0.978199
2510,2026-11-10,AAPL,Technology,348.245392,353.601307,346.196930,351.083232,29756100,-0.006756,-0.003680,...,0.140450,0.017042,0.028251,0.023223,343.971147,337.010734,1.012426,1.033336,49378270.0,0.602615
2511,2026-11-11,AAPL,Technology,355.359985,355.639772,349.064750,349.284585,34453500,0.023149,0.013188,...,0.147083,0.017565,0.017625,0.023699,344.604166,337.620271,1.031212,1.052543,48597890.0,0.708951
2512,2026-11-12,AAPL,Technology,354.880005,359.369995,354.410004,357.809998,31875000,-0.003201,-0.003540,...,0.120947,0.017548,0.014763,0.023403,345.261150,338.223606,1.027860,1.049247,48203275.0,0.661262


In [14]:
print(FE.data.columns)

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D', 'Return_10D',
       'Return_20D', 'Return_60D', 'Volatility_20D', 'Volatility_5D',
       'Volatility_10D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D', 'Volume_Average_20D', 'Relative_Volume_20D'],
      dtype='str', name='Price')


In [15]:
FE.add_target_variable(5)


FE.data = FE.data.dropna().copy()

FE.data.index
FE.save_features()

FE.data

Features data saved for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-11-14   AAPL  Technology   24.280031   24.762370   23.905645   
1     2016-11-15   AAPL  Technology   24.601593   24.732514   24.383393   
2     2016-11-16   AAPL  Technology   25.263075   25.318201   24.484442   
3     2016-11-17   AAPL  Technology   25.253891   25.345765   24.996645   
4     2016-11-18   AAPL  Technology   25.279156   25.389405   25.187283   
...          ...    ...         ...         ...         ...         ...   
2509  2026-11-09   AAPL  Technology  350.423737  352.062476  344.518220   
2510  2026-11-10   AAPL  Technology  348.245392  353.601307  346.196930   
2511  2026-11-11   AAPL  Technology  355.359985  355.639772  349.064750   
2512  2026-11-12   AAPL  Technology  354.880005  359.369995  354.410004   
2513  2026-11-13   AAPL  Technology  356.269989  360.829987  355.750000   

Price        Open     Volume  
0       24.739402  204702000  
1       24.47

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D,Target_5D_Direction
60,2017-02-10,AAPL,Technology,30.477581,30.666741,30.461435,30.556015,80262000,0.003308,0.003946,...,0.004926,0.019024,28.636203,27.327385,1.064302,1.115276,130645920.0,0.614348,0.027248,1
61,2017-02-13,AAPL,Technology,30.747471,30.869735,30.622905,30.699030,92141600,0.005652,0.005442,...,0.004829,0.018611,28.806490,27.434638,1.067380,1.120754,130030620.0,0.708615,0.025584,1
62,2017-02-14,AAPL,Technology,31.146549,31.162695,30.738243,30.788993,132904800,0.003357,0.003995,...,0.005725,0.018129,28.985706,27.554605,1.074549,1.130357,129787900.0,1.024015,0.015479,1
63,2017-02-15,AAPL,Technology,31.259592,31.434912,31.054286,31.261901,142492400,0.005913,0.005221,...,0.005750,0.004957,29.170688,27.674948,1.071610,1.129527,132169920.0,1.078100,0.007527,1
64,2017-02-16,AAPL,Technology,31.222681,31.349553,31.105031,31.296497,90338400,-0.000310,-0.000851,...,0.006512,0.004873,29.356238,27.798183,1.063579,1.123191,131567380.0,0.686632,0.009679,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2504,2026-11-02,AAPL,Technology,338.063141,343.538961,335.565039,342.409844,64757700,-0.013613,-0.004790,...,0.014387,0.012524,342.095076,333.366903,0.988214,1.014087,41928535.0,1.544478,0.036563,1
2505,2026-11-03,AAPL,Technology,331.648010,334.246042,325.082991,325.332801,63160000,0.006152,0.002992,...,0.013804,0.013500,342.008641,333.797776,0.969707,0.993560,43564085.0,1.449818,0.050045,1
2506,2026-11-04,AAPL,Technology,349.244659,354.230861,340.731108,341.310685,89637700,0.009733,0.016518,...,0.030872,0.021846,342.650154,334.589375,1.019246,1.043801,46338575.0,1.934408,0.017510,1
2507,2026-11-05,AAPL,Technology,343.908691,355.110171,341.800278,353.621312,75678900,-0.064889,-0.044092,...,0.031091,0.022114,342.837511,335.203308,1.003124,1.025970,48355900.0,1.565040,0.031902,1


In [16]:
FE.data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D,Target_5D_Direction
60,2017-02-10,AAPL,Technology,30.477581,30.666741,30.461435,30.556015,80262000,0.003308,0.003946,...,0.004926,0.019024,28.636203,27.327385,1.064302,1.115276,130645920.0,0.614348,0.027248,1
61,2017-02-13,AAPL,Technology,30.747471,30.869735,30.622905,30.699030,92141600,0.005652,0.005442,...,0.004829,0.018611,28.806490,27.434638,1.067380,1.120754,130030620.0,0.708615,0.025584,1
62,2017-02-14,AAPL,Technology,31.146549,31.162695,30.738243,30.788993,132904800,0.003357,0.003995,...,0.005725,0.018129,28.985706,27.554605,1.074549,1.130357,129787900.0,1.024015,0.015479,1
63,2017-02-15,AAPL,Technology,31.259592,31.434912,31.054286,31.261901,142492400,0.005913,0.005221,...,0.005750,0.004957,29.170688,27.674948,1.071610,1.129527,132169920.0,1.078100,0.007527,1
64,2017-02-16,AAPL,Technology,31.222681,31.349553,31.105031,31.296497,90338400,-0.000310,-0.000851,...,0.006512,0.004873,29.356238,27.798183,1.063579,1.123191,131567380.0,0.686632,0.009679,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2504,2026-11-02,AAPL,Technology,338.063141,343.538961,335.565039,342.409844,64757700,-0.013613,-0.004790,...,0.014387,0.012524,342.095076,333.366903,0.988214,1.014087,41928535.0,1.544478,0.036563,1
2505,2026-11-03,AAPL,Technology,331.648010,334.246042,325.082991,325.332801,63160000,0.006152,0.002992,...,0.013804,0.013500,342.008641,333.797776,0.969707,0.993560,43564085.0,1.449818,0.050045,1
2506,2026-11-04,AAPL,Technology,349.244659,354.230861,340.731108,341.310685,89637700,0.009733,0.016518,...,0.030872,0.021846,342.650154,334.589375,1.019246,1.043801,46338575.0,1.934408,0.017510,1
2507,2026-11-05,AAPL,Technology,343.908691,355.110171,341.800278,353.621312,75678900,-0.064889,-0.044092,...,0.031091,0.022114,342.837511,335.203308,1.003124,1.025970,48355900.0,1.565040,0.031902,1


In [17]:
data = FE.load_features()

data.columns

Data loaded for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-11-14   AAPL  Technology   24.280031   24.762370   23.905645   
1     2016-11-15   AAPL  Technology   24.601593   24.732514   24.383393   
2     2016-11-16   AAPL  Technology   25.263075   25.318201   24.484442   
3     2016-11-17   AAPL  Technology   25.253891   25.345765   24.996645   
4     2016-11-18   AAPL  Technology   25.279156   25.389405   25.187283   
...          ...    ...         ...         ...         ...         ...   
2509  2026-11-09   AAPL  Technology  350.423737  352.062476  344.518220   
2510  2026-11-10   AAPL  Technology  348.245392  353.601307  346.196930   
2511  2026-11-11   AAPL  Technology  355.359985  355.639772  349.064750   
2512  2026-11-12   AAPL  Technology  354.880005  359.369995  354.410004   
2513  2026-11-13   AAPL  Technology  356.269989  360.829987  355.750000   

Price        Open     Volume  
0       24.739402  204702000  
1       24.477563  12

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D', 'Return_10D',
       'Return_20D', 'Return_60D', 'Volatility_20D', 'Volatility_5D',
       'Volatility_10D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D', 'Volume_Average_20D', 'Relative_Volume_20D',
       'Future_Return_5D', 'Target_5D_Direction'],
      dtype='str', name='Price')

In [18]:


data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume,QQQ_Return,SPY_Return,...,Volatility_5D,Volatility_10D,SMA_20D,SMA_50D,Price_to_SMA_20D,Price_to_SMA_50D,Volume_Average_20D,Relative_Volume_20D,Future_Return_5D,Target_5D_Direction
60,2017-02-10,AAPL,Technology,30.477581,30.666741,30.461435,30.556015,80262000,0.003308,0.003946,...,0.004926,0.019024,28.636203,27.327385,1.064302,1.115276,130645920.0,0.614348,0.027248,1
61,2017-02-13,AAPL,Technology,30.747471,30.869735,30.622905,30.699030,92141600,0.005652,0.005442,...,0.004829,0.018611,28.806490,27.434638,1.067380,1.120754,130030620.0,0.708615,0.025584,1
62,2017-02-14,AAPL,Technology,31.146549,31.162695,30.738243,30.788993,132904800,0.003357,0.003995,...,0.005725,0.018129,28.985706,27.554605,1.074549,1.130357,129787900.0,1.024015,0.015479,1
63,2017-02-15,AAPL,Technology,31.259592,31.434912,31.054286,31.261901,142492400,0.005913,0.005221,...,0.005750,0.004957,29.170688,27.674948,1.071610,1.129527,132169920.0,1.078100,0.007527,1
64,2017-02-16,AAPL,Technology,31.222681,31.349553,31.105031,31.296497,90338400,-0.000310,-0.000851,...,0.006512,0.004873,29.356238,27.798183,1.063579,1.123191,131567380.0,0.686632,0.009679,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2504,2026-11-02,AAPL,Technology,338.063141,343.538961,335.565039,342.409844,64757700,-0.013613,-0.004790,...,0.014387,0.012524,342.095076,333.366903,0.988214,1.014087,41928535.0,1.544478,0.036563,1
2505,2026-11-03,AAPL,Technology,331.648010,334.246042,325.082991,325.332801,63160000,0.006152,0.002992,...,0.013804,0.013500,342.008641,333.797776,0.969707,0.993560,43564085.0,1.449818,0.050045,1
2506,2026-11-04,AAPL,Technology,349.244659,354.230861,340.731108,341.310685,89637700,0.009733,0.016518,...,0.030872,0.021846,342.650154,334.589375,1.019246,1.043801,46338575.0,1.934408,0.017510,1
2507,2026-11-05,AAPL,Technology,343.908691,355.110171,341.800278,353.621312,75678900,-0.064889,-0.044092,...,0.031091,0.022114,342.837511,335.203308,1.003124,1.025970,48355900.0,1.565040,0.031902,1


In [19]:
data.isna().sum()

Price
Date                   0
Ticker                 0
Sector                 0
Close                  0
High                   0
Low                    0
Open                   0
Volume                 0
QQQ_Return             0
SPY_Return             0
Daily_Return           0
Return_5D              0
Return_10D             0
Return_20D             0
Return_60D             0
Volatility_20D         0
Volatility_5D          0
Volatility_10D         0
SMA_20D                0
SMA_50D                0
Price_to_SMA_20D       0
Price_to_SMA_50D       0
Volume_Average_20D     0
Relative_Volume_20D    0
Future_Return_5D       0
Target_5D_Direction    0
dtype: int64